In [1]:
using Turing, StatsPlots, Distributions, CSV, DataFrames, Optim

# Load the observed dataset
data = CSV.read("../../data/ice_cream_sales.csv", DataFrame);
names(data)

┌ Warning: The call to compilecache failed to create a usable precompiled cache file for NamedArrays [86f7a689-2022-50b4-a561-43c23ac3c673]
│   exception = ErrorException("Required dependency DelimitedFiles [8bb1440f-4735-579b-a4ab-409b98df4dab] failed to load from a cache file.")
└ @ Base loading.jl:1818


6-element Vector{String}:
 "ID"
 "Temperature"
 "Is_Weekend"
 "Hours_Open"
 "Electricity_Usage"
 "Ice_Cream_Sales"

In [2]:
describe(data)

Row,variable,mean,min,median,max,nmissing,eltype
,Symbol,Float64,Real,Float64,Real,Int64,DataType
1,ID,180.5,1,180.5,360,0,Int64
2,Temperature,24.7859,19.2983,24.8258,29.7834,0,Float64
3,Is_Weekend,0.283333,false,0.0,true,0,Bool
4,Hours_Open,9.40833,8,10.0,11,0,Int64
5,Electricity_Usage,88.1732,65.6504,87.9184,115.253,0,Float64
6,Ice_Cream_Sales,1028.59,803,1025.5,1273,0,Int64


In [3]:
#= Convert continous data to Float64
And the categorical data to integers =#

data[!, :Temperature] = convert(Array{Float64}, data[!, :Temperature])
data[!, :Is_Weekend] = convert(Array{Int}, data[!, :Is_Weekend])
data[!, :Hours_Open] = convert(Array{Int}, data[!, :Hours_Open])
data[!, :Ice_Cream_Sales] = convert(Array{Float64}, data[!, :Ice_Cream_Sales])
data[!, :Electricity_Usage] = convert(Array{Float64}, data[!, :Electricity_Usage])
temperature = data[!, :Temperature]
is_weekend = data[!, :Is_Weekend]
hours_open = data[!, :Hours_Open]
ice_cream_sales = data[!, :Ice_Cream_Sales]
electricity_usage = data[!, :Electricity_Usage];

In [4]:
# Build the model

@model function ice_cream_sales_model(temperature, is_weekend, hours_open, electricity_usage, ice_cream_sales)
    n = min(length(temperature), length(is_weekend), length(hours_open), length(electricity_usage), length(ice_cream_sales))
    
    for i in 1:n
        # Priors for independent variables
        temperature[i] ~ Normal(25, 2)
        is_weekend[i] ~ Bernoulli(2/7)

        # How each model is dependent on it's predecessors
        if is_weekend[i] == 1
            hours_open[i] ~ DiscreteUniform(10,11)
        else
            hours_open[i] ~ DiscreteUniform(8,10)
        end

        electricity_usage[i] ~ Normal(10 + temperature[i] * 2 + hours_open[i] * 3, 1)
        
        # How the dependent variable is dependent on the independent variables
        ice_cream_sales[i] ~ Normal(20 + temperature[i] * 30 + hours_open[i] * 25 + is_weekend[i] * 100, 3)
    end
end

ice_cream_sales_model (generic function with 2 methods)

In [5]:
cont_sampler = HMC(0.01, 10, :temperature, :electricity_usage, :ice_cream_sales)
disc_sampler = PG(10, :is_weekend, :hours_open)
sampler = Gibbs(cont_sampler, disc_sampler)

Gibbs{(:temperature, :electricity_usage, :ice_cream_sales, :is_weekend, :hours_open), Tuple{HMC{Turing.Essential.ForwardDiffAD{0}, (:temperature, :electricity_usage, :ice_cream_sales), AdvancedHMC.UnitEuclideanMetric}, PG{(:is_weekend, :hours_open), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}}}((HMC{Turing.Essential.ForwardDiffAD{0}, (:temperature, :electricity_usage, :ice_cream_sales), AdvancedHMC.UnitEuclideanMetric}(0.01, 10), PG{(:is_weekend, :hours_open), AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}}(10, AdvancedPS.ResampleWithESSThreshold{typeof(AdvancedPS.resample_systematic), Float64}(AdvancedPS.resample_systematic, 0.5))))

In [6]:
model = ice_cream_sales_model(repeat([30], length(temperature)), is_weekend, hours_open, repeat([missing], length(temperature)), repeat([missing], length(temperature)))

DynamicPPL.Model{typeof(ice_cream_sales_model), (:temperature, :is_weekend, :hours_open, :electricity_usage, :ice_cream_sales), (), (), Tuple{Vector{Int64}, Vector{Int64}, Vector{Int64}, Vector{Missing}, Vector{Missing}}, Tuple{}, DynamicPPL.DefaultContext}(ice_cream_sales_model, (temperature = [30, 30, 30, 30, 30, 30, 30, 30, 30, 30  …  30, 30, 30, 30, 30, 30, 30, 30, 30, 30], is_weekend = [0, 0, 0, 0, 0, 1, 1, 0, 0, 0  …  0, 0, 0, 0, 0, 1, 1, 0, 0, 0], hours_open = [10, 9, 8, 10, 8, 11, 11, 8, 9, 10  …  9, 8, 10, 10, 9, 11, 11, 9, 8, 10], electricity_usage = [missing, missing, missing, missing, missing, missing, missing, missing, missing, missing  …  missing, missing, missing, missing, missing, missing, missing, missing, missing, missing], ice_cream_sales = [missing, missing, missing, missing, missing, missing, missing, missing, missing, missing  …  missing, missing, missing, missing, missing, missing, missing, missing, missing, missing]), NamedTuple(), DynamicPPL.DefaultContext())

In [7]:
chain = sample(model, sampler, MCMCThreads(), 500, 6)

Sampling (6 threads)   0%|                              |  ETA: N/A
Sampling (6 threads)  17%|█████                         |  ETA: 0:54:26
Sampling (6 threads)  33%|██████████                    |  ETA: 0:21:47
Sampling (6 threads)  50%|███████████████               |  ETA: 0:10:53
Sampling (6 threads)  67%|████████████████████          |  ETA: 0:05:27
Sampling (6 threads)  83%|█████████████████████████     |  ETA: 0:02:11
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:20:59
Sampling (6 threads) 100%|██████████████████████████████| Time: 0:21:00


Chains MCMC chain (500×721×6 Array{Float64, 3}):

Iterations        = 1:1:500
Number of chains  = 6
Samples per chain = 500
Wall duration     = 1247.78 seconds
Compute duration  = 4452.36 seconds
parameters        = electricity_usage[1], electricity_usage[2], electricity_usage[3], electricity_usage[4], electricity_usage[5], electricity_usage[6], electricity_usage[7], electricity_usage[8], electricity_usage[9], electricity_usage[10], electricity_usage[11], electricity_usage[12], electricity_usage[13], electricity_usage[14], electricity_usage[15], electricity_usage[16], electricity_usage[17], electricity_usage[18], electricity_usage[19], electricity_usage[20], electricity_usage[21], electricity_usage[22], electricity_usage[23], electricity_usage[24], electricity_usage[25], electricity_usage[26], electricity_usage[27], electricity_usage[28], electricity_usage[29], electricity_usage[30], electricity_usage[31], electricity_usage[32], electricity_usage[33], electricity_usage[34], electricity

In [8]:
function calculate_mse(chain, ice_cream_sales)
    mse = 0
    for i in 1:length(ice_cream_sales)
        difference = mean(chain["ice_cream_sales[$i]"]) - ice_cream_sales[i]
        difference = difference^2
        mse += difference
    end
    mse = mse / length(ice_cream_sales)
    return mse
end

calculate_mse (generic function with 1 method)

In [9]:
sqrt(calculate_mse(chain, ice_cream_sales))

166.73298469848226